In [29]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

In [30]:
# ...existing code...
# --- New Cell for G2 vs NS3 Comparison ---

import re

# --- Configuration for G2 vs NS3 Comparison ---

# Define the base folder containing all the run directories from all workloads
# We will process all workloads ('toy_all_to_all_one_collective', 'toy_all_reduce_one_collective', 'model')
base_comparison_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos'

# Choose what to plot: 'avg' for mean, or 'max' for maximum value
comparison_plot_metric = 'max'  # Can be 'avg' or 'max'

# --- Data Collection Logic ---

all_run_folders = []
for workload_folder in os.listdir(base_comparison_folder):
    workload_path = os.path.join(base_comparison_folder, workload_folder)
    if os.path.isdir(workload_path):
        run_folders = [os.path.join(workload_path, d) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))]
        all_run_folders.extend(run_folders)

comparison_results = []

# Regex to extract topology index
topo_idx_regex = re.compile(r'topology(\d+)(?:\.json)?$')

for folder in sorted(all_run_folders):
    run_summary_path = os.path.join(folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    # 1. Parse run_summary.txt to identify sim_type and topology
    summary_params = parse_config(run_summary_path)
    
    sim_type = None
    topo_file = None
    workload_name = summary_params.get('collective', 'N/A').strip()

    if os.path.exists(os.path.join(folder, 'g2')):
        sim_type = 'G2'
        topo_file = summary_params.get('g2 topology file override', 'N/A')
    elif os.path.exists(os.path.join(folder, 'ns3')):
        sim_type = 'NS3'
        topo_file = summary_params.get('ns3 topology file override', 'N/A')
    
    if not sim_type or not topo_file or 'all_paths' in topo_file:
        continue # Skip if not G2/NS3, no topo file, or it's a fixed 'all_paths' topology

    # Extract topology index
    match = topo_idx_regex.search(topo_file)
    if not match:
        continue
    topo_index = int(match.group(1))

    # 2. Get timing data
    timing_file = None
    sim_output_dir = None
    if sim_type == 'G2':
        sim_output_dir = os.path.join(folder, 'g2')
    elif sim_type == 'NS3':
        sim_output_dir = os.path.join(folder, 'ns3')

    if sim_output_dir and os.path.isdir(sim_output_dir):
        for f in os.listdir(sim_output_dir):
            if 'trace_matched_timing.csv' in f:
                timing_file = os.path.join(sim_output_dir, f)
                break
    
    if not timing_file:
        continue

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'NS3':
            df = df[df['node_name'] != 'dummy_node'].copy()
        
        time_col = 'callback_tick'
        if time_col not in df.columns:
            continue
            
        elapsed_times = df[time_col].dropna()
        if elapsed_times.empty:
            continue

        # 3. Create a descriptive name and store results
        run_name = f"{sim_type}"
        if sim_type == 'NS3':
            ns3_config_file = find_config_file(folder)
            if ns3_config_file:
                ns3_params = parse_config(ns3_config_file)
                run_name = (
                    f"NS3 (cc:{ns3_params.get('cc_mode', 'N/A')}, "
                    f"win:{ns3_params.get('has_win', 'N/A')}, "
                    f"adapt:{ns3_params.get('var_win', 'N/A')}, "
                    f"buf:{ns3_params.get('buffer_size', 'N/A')}, "
                    f"size:{ns3_params.get('packet_payload_size', 'N/A')})"
                )

        comparison_results.append({
            'workload': workload_name,
            'topo_index': topo_index,
            'sim_type': sim_type,
            'run_name': run_name,
            'avg_time': elapsed_times.mean(),
            'max_time': elapsed_times.max(),
            'std_dev': elapsed_times.std(),
            'path': os.path.basename(folder)
        })

    except Exception as e:
        print(f"Error processing {folder}: {e}")

# --- Plotting Logic ---

if comparison_results:
    comp_df = pd.DataFrame(comparison_results)
    
    # Determine which column to use for plotting
    if comparison_plot_metric == 'max':
        y_col = 'max_time'
        y_axis_title = "Maximum Time (ns)"
    else: # Default to 'avg'
        y_col = 'avg_time'
        y_axis_title = "Average Time (ns)"

    workloads = comp_df['workload'].unique()
    
    for wl in sorted(workloads):
        for topo_idx in sorted(comp_df['topo_index'].unique()):
            
            group_df = comp_df[(comp_df['workload'] == wl) & (comp_df['topo_index'] == topo_idx)].copy()
            
            if group_df.empty:
                continue

            # Separate G2 and NS3 for plotting
            g2_runs = group_df[group_df['sim_type'] == 'G2']
            ns3_runs = group_df[group_df['sim_type'] == 'NS3'].sort_values(by=y_col)
            
            plot_title = f'G2 vs NS3 Comparison for Workload: "{wl}", Topology: {topo_idx}'

            fig = go.Figure()

            # Add NS3 runs as bars
            fig.add_trace(go.Bar(
                x=ns3_runs['run_name'],
                y=ns3_runs[y_col],
                name='NS3 Runs',
                marker_color='rgb(55, 83, 109)',
                text=ns3_runs[y_col].apply(lambda x: f'{x/1e9:.4f} s'),
                textposition='outside'
            ))

            # Add G2 run as a benchmark line
            if not g2_runs.empty:
                g2_time = g2_runs.iloc[0][y_col]
                fig.add_hline(
                    y=g2_time, 
                    line_dash="dot",
                    annotation_text=f"G2 Time: {g2_time/1e9:.4f} s", 
                    annotation_position="top right",
                    line_color="red",
                    annotation=dict(
                        font=dict(color="white", size=12),
                        bgcolor="red",
                        borderpad=4
                    )
                )

            fig.update_layout(
                title=plot_title,
                xaxis_title="Run Configuration",
                yaxis_title=y_axis_title,
                xaxis={'tickangle': -60},
                template='plotly_white',
                height=700,
                width=1400,
                margin=dict(b=350),
                showlegend=True
            )
            
            print(f"\n--- Plot for Workload: {wl}, Topology: {topo_idx} ---")
            fig.show()
else:
    print("\nNo comparison results to plot.")

# Display the full data table
if comparison_results:
    print("\n--- Full Comparison Data ---")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
        display(comp_df.sort_values(by=['workload', 'topo_index', y_col]))



--- Plot for Workload: model, Topology: 0 ---



--- Plot for Workload: model, Topology: 1 ---



--- Plot for Workload: model, Topology: 2 ---



--- Plot for Workload: model, Topology: 3 ---



--- Plot for Workload: model, Topology: 4 ---


...


--- Plot for Workload: toy_all_to_all_one_collective, Topology: 2 ---



--- Plot for Workload: toy_all_to_all_one_collective, Topology: 3 ---



--- Plot for Workload: toy_all_to_all_one_collective, Topology: 4 ---



--- Full Comparison Data ---


,workload,topo_index,sim_type,run_name,avg_time,max_time,std_dev,path
15,model,0,NS3,"NS3 (cc:8, win:1, adapt:1, buf:1, size:1500)",2.700575e+10,57059773757,1.438841e+10,run_20251120_065321
19,model,0,NS3,"NS3 (cc:8, win:1, adapt:1, buf:8, size:1500)",2.700575e+10,57059773757,1.438841e+10,run_20251120_070156
5,model,0,NS3,"NS3 (cc:10, win:1, adapt:1, buf:1, size:1500)",3.126469e+10,58489422802,1.458590e+10,run_20251120_064411
10,model,0,NS3,"NS3 (cc:10, win:1, adapt:1, buf:8, size:1500)",3.126469e+10,58489422802,1.458590e+10,run_20251120_064846
58,model,0,G2,G2,3.149240e+10,58605521161,1.480035e+10,run_20251120_111121_988ms
53,model,0,NS3,"NS3 (cc:0, win:0, adapt:0, buf:8, size:1500)",3.803449e+10,65830548733,1.615631e+10,run_20251120_082057
23,model,0,NS3,"NS3 (cc:1, win:1, adapt:1, buf:1, size:1500)",3.317079e+10,66503080774,1.601447e+10,run_20251120_071025
28,model,0,NS3,"NS3 (cc:1, win:1, adapt:1, buf:8, size:1500)",3.317079e+10,66503080774,1.601447e+10,run_20251120_072410
2,model,0,NS3,"NS3 (cc:3, win:1, adapt:1, buf:8, size:1500)",2.166932e+11,382731638602,8.786914e+10,run_20251120_063115
33,model,0,NS3,"NS3 (cc:7, win:0, adapt:0, buf:1, size:1500)",2.087005e+11,385102841376,8.993239e+10,run_20251120_073743
